In [1]:
import pandas as pd
import statsmodels.api as sm
from arch import arch_model
from data_wrangling import *

## Pulling data from various data files

In [2]:
# data_dxy = yf.download("DX-Y.NYB", start="2000-01-01", end="2024-01-01", threads=False)
# data_2yr = yf.download("^IRX", start="2000-01-01", end="2024-01-01", threads=False)
# data_10yr = yf.download("^TNX", start="2000-01-01", end="2024-01-01", threads=False)
# data_vix = yf.download("^VIX", start="2000-01-01", end="2024-01-01", threads=False)

data_dxy = pd.read_csv("data/data_dxy.csv", index_col=0).Close
data_2yr = pd.read_csv("data/data_2yr.csv", index_col=0).Close
data_10yr = pd.read_csv("data/data_10yr.csv", index_col=0).Close
data_vix = pd.read_csv("data/data_vix.csv", index_col=0).Close

data_2s10s = data_2yr - data_10yr

aapl_time_series = pd.read_hdf("data/all_tickers_time_series.hf5", key="AAPL")

## EDA

In [3]:
adf_test(data_dxy)
print('---')
adf_test(data_2s10s)
print('---')
adf_test(data_vix)

ADF Statistic: -1.5382624377815282
p-value: 0.5145057565985448
Non-stationary: Consider differencing or other transformations
---
ADF Statistic: -1.495927085225662
p-value: 0.5355251808176756
Non-stationary: Consider differencing or other transformations
---
ADF Statistic: -5.810247495128666
p-value: 4.4186299154603377e-07
Stationary: No differencing required


In [7]:
def goldfeld_quandt_test(series_x: pd.Series, series_y: pd.Series, split=None) -> dict:
    """
    Perform Goldfeld-Quandt test.
    """
    exog_var = sm.add_constant(series_x)
    model = sm.OLS(series_y, exog_var.astype(float), missing='drop').fit()
    result = sm.stats.diagnostic.het_goldfeldquandt(model.resid, model.model.exog, split=split)
    test_results = dict(lzip(['F-statistic', 'p-value'], result))
    
    return test_results

aapl_time_series
# goldfeld_quandt_test(aapl_time_series['prc'].pct_change(), data_dxy)

,ticker,date,openprc,askhi,bidlo,prc,vol,dlstdt,dlstcd,nextdt,dlprc
0,AAPL,2000-01-03,104.87500,112.50000,101.68750,111.93750,4833736.0,2023-12-29,100,NaT,0.0
1,AAPL,2000-01-04,108.25000,110.62500,101.18750,102.50000,4646120.0,2023-12-29,100,NaT,0.0
2,AAPL,2000-01-05,103.75000,110.56250,103.00000,104.00000,7060334.0,2023-12-29,100,NaT,0.0
3,AAPL,2000-01-06,106.12500,107.00000,95.00000,95.00000,6917851.0,2023-12-29,100,NaT,0.0
4,AAPL,2000-01-07,96.50000,101.00000,95.50000,99.50000,4180715.0,2023-12-29,100,NaT,0.0
...,...,...,...,...,...,...,...,...,...,...,...
30180,AAPL,2023-12-22,195.17999,195.41000,192.97000,193.60001,36702455.0,2023-12-29,100,NaT,0.0
30181,AAPL,2023-12-26,193.61000,193.89000,192.83000,193.05000,28541150.0,2023-12-29,100,NaT,0.0
30182,AAPL,2023-12-27,192.49001,193.50000,191.09000,193.14999,47538658.0,2023-12-29,100,NaT,0.0
30183,AAPL,2023-12-28,194.14000,194.66000,193.17000,193.58000,33691704.0,2023-12-29,100,NaT,0.0
